# The Paired t-Test: Two Groups That Are Really One

The two-sample test assumed its groups were independent — but weigh ten people *before* a training program and weigh the *same ten people* after, and the groups are *paired*: each "before" has its personal "after." The right move is to study the **differences** $d_i = x_i - y_i$ person by person. If the populations are Gaussian, so are the differences — and testing "no weight change" is just $H_0: \mu_d = 0$, attacked with the familiar one-sample machinery: $\bar{d} = 1.09$, $s_d = 1.485$, $t_{obs} = 2.321$ against $t_9$, giving $p = 0.0227 < 0.05$ — the program works. The closing insight is the best part: once the $d_i$ are defined, **the paired test *is* a one-sample t-test** — everything already learned applies verbatim. It is based on the eleventh video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

Not all two-group comparisons involve two populations. Before/after, left/right, twin A/twin B — whenever each measurement in one group has a partner in the other, independence fails *by design*, and the fix is beautifully simple: subtract within pairs and test the differences. This notebook builds the paired t-test, reproduces the video's weight-loss verdict, and quantifies (in the exercises) why pairing is often a massive statistical bargain.

After completing this notebook you will be able to:

- Recognize paired data and say precisely which two-sample assumption it violates.
- Define the differences $d_i$, their mean $\bar{D}$, and the paired t-statistic with $n - 1$ degrees of freedom.
- Reproduce the video's test: $\bar{d} = 1.09$, $s_d = 1.485$, $t = 2.321$, $p = 0.0227$ → the program works.
- Exploit the reduction: run the paired test as a plain one-sample t-test on the differences.
- Show what pairing buys — and what analyzing the same data as independent groups would have thrown away.

## 2. Theory

### When two groups aren't independent

You want to test a four-week training program for weight loss. Group one: the ten participants *before* the program. Group two: the **same ten people** after. Each "before" measurement corresponds to a specific "after" — the samples are **paired**, and the two-sample test's independence assumption is dead on arrival.

### Study the differences

What you actually care about is the change within each person. Define a new variable per pair:

$$
d_1 = x_1 - y_1, \quad d_2 = x_2 - y_2, \quad \dots, \quad d_{10} = x_{10} - y_{10}
$$

(weight before minus weight after — positive $d$ = weight lost). The average behavior is captured by the sample mean of the differences,

$$
\bar{D} = \frac{1}{10}\sum_{i=1}^{10} D_i
$$

### The statistic — an old friend

If $X$ and $Y$ come from Gaussian populations, $D$ is Gaussian too (a difference of Gaussians). Standardizing $\bar{D}$ would give a standard normal — but the population standard deviation of $D$ is unknown, so, exactly as before, replace it with the sample standard deviation $S_D$:

$$
T = \frac{\bar{D} - \mu_D}{S_D/\sqrt{n}} \;\sim\; t_{n-1} = t_9
$$

Whatever tail you choose, the null hypothesis is always **no weight difference**: $\mu_D = 0$, so under $H_0$ the statistic is simply $\bar{D}/(S_D/\sqrt{n})$.

### The observed test

Subtracting the ten after-weights from the ten before-weights, one by one, gives the observed differences, and from them:

$$
\bar{d} = 1.09, \qquad s_d = 1.485, \qquad
t_{obs} = \frac{1.09}{1.485/\sqrt{10}} = 2.321
$$

**Right-tailed test** ($H_1: \mu_D > 0$ — "the program helps you lose weight"), $\alpha = 0.05$:

$$
p = P(T > 2.321 \mid H_0) = 0.0227 < 0.05
$$

Reject $H_0$: the population mean weight difference is positive — the training program is good for losing weight.

### The punchline

Look closely: once you work with the $d_i$, the whole problem **reduces to a single sample of size $n$** — one list of numbers, one mean, one standard deviation, $H_0$ about one parameter. *All the results you learned for one-sample t-tests are valid here.* The paired test isn't a new test; it's a change of variables that turns a dependent two-group problem into the simplest problem in the lesson.

## 3. Worked Examples

### Example 1: the arithmetic

$s_d/\sqrt{n} = 1.485/\sqrt{10} = 0.470$; $t_{obs} = 1.09/0.470 = 2.321$; df $= 10 - 1 = 9$; right tail of $t_9$ beyond 2.321: $0.0227$.

### Example 2: reading the three alternatives

With $d = \text{before} - \text{after}$: $H_1: \mu_D > 0$ — the program *reduces* weight (the claim to prove); $H_1: \mu_D < 0$ — the program *adds* weight; $H_1: \mu_D \ne 0$ — the program changes weight either way. Note the sign convention is a modeling choice: had we defined $d = \text{after} - \text{before}$, the tails would swap — decide the convention *before* looking at the data, and write it down.

### Example 3: why not the two-sample test?

The two-sample statistic's standard error is built from the *total* spread of each group — which includes the person-to-person differences in baseline weight (some participants weigh 78 kg, some 92 kg). Pairing subtracts each person's baseline from themselves, deleting that between-person variance from the problem and leaving only the *changes*, which are far less variable. Same data, dramatically sharper ruler — Exercise 4 puts numbers on it.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project) and the t-machinery.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import gamma, sqrt, pi

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(21)

def t_pdf(t, df):
    """Density of the Student t-distribution with df degrees of freedom."""
    coeff = gamma((df + 1) / 2) / (sqrt(df * pi) * gamma(df / 2))
    return coeff * (1 + np.asarray(t, dtype=float) ** 2 / df) ** (-(df + 1) / 2)

def t_tail(t_value, df):
    """P(T > t_value) for T ~ t_df, by numeric integration."""
    grid = np.linspace(0, abs(t_value), 20_000)
    cdf_half = np.trapezoid(t_pdf(grid, df), grid)
    tail = 0.5 - cdf_half
    return tail if t_value >= 0 else 1 - tail

def one_sample_t_test(data, mu0=0.0, tail="right", alpha=0.05, label=""):
    """The one-sample t-test -- which, on differences, IS the paired test."""
    n = len(data)
    t_obs = (np.mean(data) - mu0) / (np.std(data, ddof=1) / np.sqrt(n))
    p = {"right": t_tail(t_obs, n - 1),
         "left": 1 - t_tail(t_obs, n - 1),
         "two": 2 * t_tail(abs(t_obs), n - 1)}[tail]
    verdict = "REJECT H0" if p < alpha else "fail to reject H0"
    print(f"{label}: t_obs = {t_obs:.3f} (df = {n - 1}), p = {p:.4f} -> {verdict}")
    return t_obs, p

### The ten participants, before and after

In [ ]:
before = np.array([86.6, 82.5, 83.0, 80.3, 77.6, 82.3, 88.4, 83.3, 91.7, 83.6])
after = np.array([83.7, 83.5, 81.5, 79.6, 76.8, 81.1, 89.0, 82.3, 91.3, 79.6])
d = before - after

print("differences d_i = before - after (kg):")
print(" ", np.round(d, 1))
print(f"\nd-bar = {d.mean():.2f} kg, s_d = {d.std(ddof=1):.3f}, n = {len(d)}")
print("(two participants -- persons 2 and 7 -- actually GAINED weight; the test")
print(" asks about the average, not every individual)")

### The paired test — run as the one-sample test it secretly is

In [ ]:
t_obs, p = one_sample_t_test(d, mu0=0, tail="right",
                             label="paired right-tailed (program helps)")
print(f"\nConclusion: the mean weight difference is positive -- the training")
print(f"program is good for losing weight (p = {p:.4f} < 0.05).")

_ = one_sample_t_test(d, mu0=0, tail="two",
                      label="paired two-tailed  (program changes weight)")

### The payoff of pairing: the wrong analysis, for contrast

Feed the *same* twenty numbers into the independent two-sample (Welch) test — as if before and after were different people — and watch the evidence drown in between-person variance.

In [ ]:
nx = ny = len(before)
vx, vy = before.var(ddof=1) / nx, after.var(ddof=1) / ny
se_indep = np.sqrt(vx + vy)
t_indep = (before.mean() - after.mean()) / se_indep
df_w = (vx + vy) ** 2 / (vx**2 / (nx - 1) + vy**2 / (ny - 1))
p_indep = t_tail(t_indep, df_w)

print(f"same mean difference: {before.mean() - after.mean():.2f} kg either way")
print(f"paired SE:      {d.std(ddof=1) / np.sqrt(10):.3f}  -> t = {t_obs:.3f}, "
      f"p = {p:.4f}  (REJECT)")
print(f"independent SE: {se_indep:.3f}  -> t = {t_indep:.3f}, "
      f"p = {p_indep:.4f}  (nowhere near)")
print("\nThe independent test's SE must carry the ~4 kg person-to-person spread;")
print("pairing subtracts each person from themselves and keeps only the changes.")

## 5. Visualizations

### The pairs themselves

One line per participant — most slope downward (weight lost), two rise. The paired test lives entirely in these slopes.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
for b, a in zip(before, after):
    lost = b > a
    ax.plot([0, 1], [b, a], "o-", color="tab:green" if lost else "tab:red",
            alpha=0.7, markersize=5)
ax.set_xticks([0, 1])
ax.set_xticklabels(["before", "after"])
ax.set_xlim(-0.3, 1.3)
ax.set_ylabel("weight (kg)")
ax.set_title("Ten participants: 8 lost weight (green), 2 gained (red)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### The differences, and the test they feed

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

jitter = rng.normal(0, 0.03, size=len(d))
axes[0].scatter(d, jitter, s=70, color="tab:blue", alpha=0.7, edgecolor="white")
axes[0].axvline(0, color="black", linestyle="-", linewidth=1)
axes[0].axvline(d.mean(), color="tab:blue", linestyle="--", linewidth=2,
                label=f"$\\bar{{d}}$ = {d.mean():.2f}")
axes[0].set_ylim(-0.25, 0.25)
axes[0].set_yticks([])
axes[0].set_xlabel("weight difference $d_i$ (kg)")
axes[0].set_title("The reduced dataset: one sample of ten differences", fontsize=10)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis="x")

t_axis = np.linspace(-4.5, 4.5, 600)
density = t_pdf(t_axis, 9)
axes[1].plot(t_axis, density, color="black", linewidth=1.6)
axes[1].fill_between(t_axis, density, where=t_axis >= t_obs, color="tab:red",
                     alpha=0.55)
axes[1].axvline(t_obs, color="tab:red", linewidth=1.8)
axes[1].axvline(0, color="black", linestyle=":", linewidth=1)
axes[1].set_xlabel("t")
axes[1].set_title(f"$t_9$: p = P(T > {t_obs:.3f}) = {p:.4f} -> reject", fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle("A paired test is a one-sample test wearing a disguise")
plt.tight_layout()
plt.show()

### The whole story on one page

Everything above in a single sheet: the before/after weights, the differences as one sample, the t-test on them, and why pairing shrinks the standard error.

<img src="img/11_paired_t_test.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Run the left-tailed paired test ($H_1$: the program *adds* weight) and interpret its p-value. Then state all three verdicts (right, two, left) at $\alpha = 0.05$ in one sentence each.

**Exercise 2.** Paired or independent? (a) blood pressure of patients before and after a drug; (b) test scores of two different schools; (c) tread wear of the left vs right front tire across 30 cars; (d) heights of US vs Argentine 18-year-olds.

### Intermediate

**Exercise 3.** Verify the reduction claim numerically: compute the paired statistic "by hand" (from $\bar{d}$ and $s_d$) and via the `one_sample_t_test` function on `d` with $\mu_0 = 0$, and confirm every intermediate number matches.

**Exercise 4.** *A stronger claim.* Test whether the program removes **more than half a kilo** on average: $H_0: \mu_D = 0.5$ vs $H_1: \mu_D > 0.5$. Verdict at $\alpha = 0.05$? What does the pair of results (rejecting $\mu_D = 0$, this verdict) jointly tell a marketing department?

### Advanced

**Exercise 5.** *When does pairing win?* Simulate before/after data with a shared per-person baseline of standard deviation $\sigma_{person}$ and independent noise $\sigma_{noise} = 1$: for $\sigma_{person} \in \{0, 2, 4\}$, estimate the power of the paired test vs the (wrong) independent test at a true mean loss of 1 kg. Where is pairing a huge win, and where is it merely harmless?

**Exercise 6.** *The confidence-interval duality.* Build the 95% t-interval for $\mu_D$ (Lesson 7's machinery: $\bar{d} \pm t_{0.975,9}\, s_d/\sqrt{n}$) and check it against the two-tailed test's verdict. State the general duality between "the interval excludes 0" and "the two-tailed test rejects at $\alpha$"." 

## 7. Solutions

In [ ]:
# Exercise 1: the left-tailed test and the trio
_, p_left = one_sample_t_test(d, 0, "left", label="left-tailed (program adds weight)")
print("\nRight-tailed: reject -- evidence the program reduces weight on average.")
print("Two-tailed:   reject -- evidence the program changes weight (p = 0.0455).")
print("Left-tailed:  p = 0.99 -- no evidence whatsoever of weight gain, as the")
print("observed differences point the opposite way.")

In [ ]:
# Exercise 2: paired or independent?
print("(a) PAIRED -- same patients, before/after.")
print("(b) INDEPENDENT -- two distinct groups of students.")
print("(c) PAIRED -- left and right tire share the same car (and driver, roads).")
print("(d) INDEPENDENT -- the last notebook's two-sample test was the right one.")

In [ ]:
# Exercise 3: the reduction, checked
d_bar, s_d, n = d.mean(), d.std(ddof=1), len(d)
t_hand = d_bar / (s_d / np.sqrt(n))
p_hand = t_tail(t_hand, n - 1)
print(f"by hand: d-bar = {d_bar:.2f}, s_d = {s_d:.4f}, "
      f"t = {t_hand:.4f}, df = {n - 1}, p = {p_hand:.4f}")
t_func, p_func = one_sample_t_test(d, 0, "right", label="via one_sample_t_test")
print(f"\nidentical: {np.isclose(t_hand, t_func) and np.isclose(p_hand, p_func)}")
print("The paired test and the one-sample test on differences are the same test.")

In [ ]:
# Exercise 4: testing mu_D = 0.5
_, p_half = one_sample_t_test(d, mu0=0.5, tail="right",
                              label="H0: mu_D = 0.5 vs H1: mu_D > 0.5")
print("\nJointly: the data support 'the program causes SOME average weight loss'")
print("but cannot support 'more than half a kilo' -- the honest marketing claim")
print("is qualitative, not quantitative. (The next exercise's confidence interval")
print("says the same thing in interval form.)")

In [ ]:
# Exercise 5: the value of pairing vs shared baseline strength
trials, n5 = 20_000, 10
true_loss = 1.0
for sigma_person in (0, 2, 4):
    baseline = rng.normal(85, sigma_person, size=(trials, n5))
    before_s = baseline + rng.normal(0, 1, size=(trials, n5))
    after_s = baseline - true_loss + rng.normal(0, 1, size=(trials, n5))

    d_s = before_s - after_s
    t_paired = d_s.mean(axis=1) / (d_s.std(axis=1, ddof=1) / np.sqrt(n5))
    power_paired = (t_paired > 1.8331).mean()

    vx = before_s.var(axis=1, ddof=1) / n5
    vy = after_s.var(axis=1, ddof=1) / n5
    t_ind = (before_s.mean(axis=1) - after_s.mean(axis=1)) / np.sqrt(vx + vy)
    power_ind = (t_ind > 1.734).mean()          # t crit, df ~ 18

    print(f"sigma_person = {sigma_person}: paired power = {power_paired:.1%}, "
          f"independent power = {power_ind:.1%}")
print("\nWith no shared baseline the two are comparable (pairing costs a few df);")
print("as person-to-person spread grows, the independent test drowns while the")
print("paired test, blind to baselines by construction, keeps its full power.")

In [ ]:
# Exercise 6: the CI duality
t975 = 2.2622                                   # t quantile 0.975, df = 9
margin = t975 * d.std(ddof=1) / np.sqrt(len(d))
lo, hi = d.mean() - margin, d.mean() + margin
print(f"95% t-interval for mu_D: [{lo:.3f}, {hi:.3f}] kg")
print(f"excludes 0? {not (lo <= 0 <= hi)}")
_, p_two6 = one_sample_t_test(d, 0, "two", label="two-tailed test")
print("\nDuality: the 95% interval excludes 0 exactly when the two-tailed test")
print("rejects at alpha = 0.05 -- they are the same computation read two ways")
print("(here both verdicts agree, and both are close calls: the interval's lower")
print("limit sits barely above zero, just as p = 0.0455 sits barely below 0.05).")

## 8. Conclusion

- **Paired samples** — the same individuals measured twice — violate the two-sample test's independence assumption by design; the fix is to analyze the **within-pair differences** $d_i = x_i - y_i$.
- With Gaussian populations the differences are Gaussian, and the test of "no change" is $H_0: \mu_D = 0$ via $T = \bar{D}/(S_D/\sqrt{n}) \sim t_{n-1}$.
- The video's numbers: $\bar{d} = 1.09$ kg, $s_d = 1.485$, $t_{obs} = 2.321$, right-tailed $p = 0.0227 < 0.05$ → the training program is good for losing weight.
- **The punchline:** defining the $d_i$ reduces everything to a single sample of size $n$ — the paired t-test *is* the one-sample t-test, and every earlier result applies verbatim.
- What pairing buys (exercises): it deletes between-person variance from the standard error — the same data analyzed as independent groups gives $p \approx 0.29$, and simulated power collapses from 66% to 0.3% as baselines spread — plus the CI–test duality: the 95% interval for $\mu_D$, $[0.03, 2.15]$, excludes zero exactly as the two-tailed $p = 0.0455$ squeaks under 0.05.

Next: the lesson's finale — A/B testing, where all of this machinery goes to work on the web.